In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ARIMA multi-run deterministic script
Integrasi: determinism fixes (seed reset, BLAS single-thread, reset seed before .fit())
"""

import os
import random

# ==================== DETERMINISM ENV (SET BEFORE NUMPY/BLAS USE) ====================
seed_value = 42
random.seed(seed_value)
os.environ['PYTHONHASHSEED'] = str(seed_value)

# Force single-thread for BLAS / OpenMP to reduce numerical variation across runs
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

# Now import scientific libs
import numpy as np
np.random.seed(seed_value)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json
import pickle
import shutil

# ==================== KONFIGURASI ====================
ticker = "AAPL"
base_path = "saved_models_arima"
os.makedirs(base_path, exist_ok=True)

covid_periods = {
    'Keseluruhan': ('2017-04-09', '2025-09-15'),
    'Sebelum COVID': ('2017-04-09', '2020-02-29'),
    'Selama COVID': ('2020-03-01', '2022-12-31'),
    'Setelah COVID': ('2023-01-01', '2025-09-15')
}

exog_features = [
    'Sentiment_Lag1', 'MA_5', 'MA_20', 'Return_1D',
    'Volatility_5D', 'RSI_14', 'Momentum_5D',
    'MA_5_20_ratio', 'RSI_MA5'
]

# ==================== FUNGSI MAPE ====================
def mape(y_true, y_pred, epsilon=1e-6):
    y = np.maximum(np.abs(y_true), epsilon)
    return np.mean(np.abs((y_true - y_pred) / y)) * 100

# ==================== PATH HELPERS ====================
def model_path_for(period_name):
    return os.path.join(base_path, f"model_arima_{period_name.replace(' ','_').lower()}.pkl")

def scaler_path_for(period_name):
    return os.path.join(base_path, f"scaler_exog_{period_name.replace(' ','_').lower()}.pkl")

def params_path_for(period_name):
    return os.path.join(base_path, f"params_arima_{ticker.lower()}_{period_name.lower().replace(' ','_')}.json")

# ==================== MEMUAT DATA ====================
processed_features_path = os.path.join(base_path, f"processed_features_{ticker.lower()}.csv")
if not os.path.exists(processed_features_path):
    raise FileNotFoundError(f"Processed features tidak ditemukan: {processed_features_path}")
data_daily = pd.read_csv(processed_features_path, parse_dates=['Date'], index_col='Date')

# ==================== PREDICT FOR PERIOD (deterministic-aware) ====================
def predict_for_period(period_name, start_date, end_date, retrain_interval=30):
    model_path = model_path_for(period_name)
    scaler_path = scaler_path_for(period_name)
    param_path = params_path_for(period_name)
    
    # Load params/scaler/model (expected precomputed by your main training script)
    if not os.path.exists(param_path):
        raise FileNotFoundError(f"Param file tidak ditemukan: {param_path}")
    if not os.path.exists(scaler_path):
        raise FileNotFoundError(f"Scaler file tidak ditemukan: {scaler_path}")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file tidak ditemukan: {model_path}")
    
    with open(param_path, 'r') as f:
        order = tuple(json.load(f)['order'])
    with open(scaler_path, 'rb') as f:
        scaler_exog = pickle.load(f)
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    
    dfp = data_daily.loc[start_date:end_date].copy().dropna()
    
    if period_name == 'Keseluruhan':
        exog_scaled = scaler_exog.transform(dfp[exog_features])
        preds = model.predict(start=0, end=len(dfp)-1, exog=exog_scaled).values
        return dfp.index, dfp['Close'].values, preds
    else:
        # Rolling forecast
        split = int(len(dfp) * 0.8)
        train = dfp.iloc[:split]
        test = dfp.iloc[split:]
        
        train_exog_scaled = scaler_exog.transform(train[exog_features])
        test_exog_scaled = scaler_exog.transform(test[exog_features])
        
        hist = train['Close'].values.tolist()
        exog_hist = train_exog_scaled.tolist()
        cur_mod = model
        preds = []
        
        for i in range(len(test)):
            xf = test_exog_scaled[i].reshape(1, -1)
            # forecast 1-step
            f = cur_mod.forecast(steps=1, exog=xf)
            preds.append(f[0])
            hist.append(test['Close'].iloc[i])
            exog_hist.append(xf.flatten().tolist())
            
            if (i + 1) % retrain_interval == 0:
                # Reset seeds before refit to ensure deterministic optimizer starting state
                random.seed(seed_value)
                np.random.seed(seed_value)
                
                # Try using consistent solver params; fallback if signature differs
                try:
                    cur_mod = ARIMA(hist, order=order, exog=exog_hist).fit(method='lbfgs', maxiter=500, disp=0)
                except TypeError:
                    # Older/newer statsmodels may not accept same args; fallback to default
                    cur_mod = ARIMA(hist, order=order, exog=exog_hist).fit()
        
        return test.index, test['Close'].values, np.array(preds)

# ==================== RUN SINGLE EXPERIMENT ====================
def run_arima_experiment(run_idx):
    print(f"\n{'='*50}")
    print(f"🚀 MEMULAI RUN ARIMA KE-{run_idx}")
    print(f"{'='*50}")
    
    # Reset seeds at start of run to ensure identical run starts
    random.seed(seed_value)
    np.random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    run_path = os.path.join(base_path, f"run_{run_idx}")
    os.makedirs(run_path, exist_ok=True)
    export_dir = os.path.join(run_path, "export_arima_results")
    os.makedirs(export_dir, exist_ok=True)
    
    # Copy precomputed model/scaler/params ke folder run (kecuali Keseluruhan optional)
    for period_name in covid_periods.keys():
        if period_name != 'Keseluruhan':
            model_src = model_path_for(period_name)
            scaler_src = scaler_path_for(period_name)
            params_src = params_path_for(period_name)
            if os.path.exists(model_src):
                shutil.copy(model_src, os.path.join(run_path, os.path.basename(model_src)))
            if os.path.exists(scaler_src):
                shutil.copy(scaler_src, os.path.join(run_path, os.path.basename(scaler_src)))
            if os.path.exists(params_src):
                shutil.copy(params_src, os.path.join(run_path, os.path.basename(params_src)))
    
    results = {}
    metrics_dict = {}
    
    for period_name, (start_date, end_date) in covid_periods.items():
        print(f"🔨 Memproses periode {period_name}...")
        dates, actual, pred = predict_for_period(period_name, start_date, end_date)
        
        results[period_name] = {
            'dates': dates,
            'actual': actual,
            'pred': pred
        }
        
        metrics_dict[period_name] = {
            'MSE': float(mean_squared_error(actual, pred)),
            'MAE': float(mean_absolute_error(actual, pred)),
            'R2': float(r2_score(actual, pred)),
            'MAPE': float(mape(actual, pred))
        }
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
        ax.plot(dates, actual, label='Aktual')
        ax.plot(dates, pred, '--', label='Prediksi')
        ax.set_title(f'ARIMA: Prediksi vs Aktual ({period_name})')
        ax.set_xlabel('Tanggal')
        ax.set_ylabel('Harga Saham (USD)')
        ax.legend()
        ax.grid(True)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        fig.autofmt_xdate()
        
        plot_path = os.path.join(run_path, f"plot_{period_name.replace(' ', '_').lower()}.png")
        fig.savefig(plot_path, dpi=300)
        plt.close(fig)
        print(f"💾 Grafik disimpan di {plot_path}")
    
    # Save metrics & csvs
    all_metrics_path = os.path.join(export_dir, "all_metrics.json")
    with open(all_metrics_path, 'w') as f:
        json.dump(metrics_dict, f, indent=4)
    
    for period_name in covid_periods.keys():
        file_key = 'keseluruhan' if period_name == 'Keseluruhan' else period_name.lower().replace(' ', '_')
        df_out = pd.DataFrame({
            'Date': results[period_name]['dates'],
            'Actual': results[period_name]['actual'],
            'Predicted': results[period_name]['pred']
        })
        csv_path = os.path.join(export_dir, f"arima_{file_key}.csv")
        df_out.to_csv(csv_path, index=False)
        json_path = os.path.join(export_dir, f"metrics_arima_{file_key}.json")
        with open(json_path, 'w') as f:
            json.dump(metrics_dict[period_name], f, indent=4)
    
    print(f"\n✅ RUN ARIMA {run_idx} SELESAI")
    return metrics_dict, results

# ==================== MENJALANKAN 30 KALI ====================
if __name__ == "__main__":
    all_run_metrics = {}
    all_predictions = {period: [] for period in covid_periods.keys()}
    all_actuals = {period: [] for period in covid_periods.keys()}
    all_dates = {period: [] for period in covid_periods.keys()}
    
    # run 30 times (should be identical results per run with determinism in place)
    for run_idx in range(1, 31):
        # reset seeds env again at each iteration
        random.seed(seed_value)
        np.random.seed(seed_value)
        os.environ['PYTHONHASHSEED'] = str(seed_value)
        
        run_metrics, run_results = run_arima_experiment(run_idx)
        all_run_metrics[f"run_{run_idx}"] = run_metrics
        
        for period in covid_periods.keys():
            all_dates[period] = run_results[period]['dates']
            all_actuals[period].append(run_results[period]['actual'])
            all_predictions[period].append(run_results[period]['pred'])
    
    # Save summary
    summary_path = os.path.join(base_path, "all_runs_summary.json")
    with open(summary_path, 'w') as f:
        json.dump(all_run_metrics, f, indent=4)
    print("\n🎉 SEMUA 30 RUN ARIMA SELESAI!")
    print(f"💾 Summary metrik disimpan di {summary_path}")
    
    # Average visualization (same as sebelumnya)
    avg_results_path = os.path.join(base_path, "average_results")
    os.makedirs(avg_results_path, exist_ok=True)
    
    for period in covid_periods.keys():
        if not all_predictions[period]:
            continue
        avg_pred = np.nanmean(all_predictions[period], axis=0)
        actual = all_actuals[period][0]
        dates = all_dates[period]
        file_key = 'keseluruhan' if period == 'Keseluruhan' else period.lower().replace(' ', '_')
        df_avg = pd.DataFrame({'Date': dates, 'Actual': actual, 'Predicted_Avg': avg_pred})
        csv_path = os.path.join(avg_results_path, f"arima_avg_{file_key}.csv")
        df_avg.to_csv(csv_path, index=False)
        
        plt.figure(figsize=(12, 6))
        plt.plot(dates, actual, label='Aktual', linewidth=2)
        plt.plot(dates, avg_pred, '--', label='Prediksi Rata-rata', linewidth=1.5)
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=2))
        plt.gcf().autofmt_xdate()
        plt.title(f'ARIMA: Rata-rata Prediksi vs Aktual ({period})')
        plt.xlabel('Tanggal')
        plt.ylabel('Harga Saham (USD)')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plot_path = os.path.join(avg_results_path, f"arima_avg_plot_{file_key}.png")
        plt.savefig(plot_path, dpi=300)
        plt.close()
        print(f"✅ Plot rata-rata prediksi untuk {period} disimpan di {plot_path}")
    
    # Print mean/std statistik across runs
    print("\n🔍 Statistik Mean & Standard Deviation untuk ARIMA (30 run):")
    for period in covid_periods.keys():
        mse_vals = []
        mae_vals = []
        r2_vals = []
        mape_vals = []
        for runm in all_run_metrics.values():
            m = runm[period]
            mse_vals.append(m['MSE'])
            mae_vals.append(m['MAE'])
            r2_vals.append(m['R2'])
            mape_vals.append(m['MAPE'])
        mse_mean = np.mean(mse_vals); mse_std = np.std(mse_vals, ddof=1)
        mae_mean = np.mean(mae_vals); mae_std = np.std(mae_vals, ddof=1)
        r2_mean = np.mean(r2_vals); r2_std = np.std(r2_vals, ddof=1)
        mape_mean = np.mean(mape_vals); mape_std = np.std(mape_vals, ddof=1)
        print(f"\n► Periode {period}:")
        print(f"   - MSE:  mean = {mse_mean:.4f}, std = {mse_std:.4f}")
        print(f"   - MAE:  mean = {mae_mean:.4f}, std = {mae_std:.4f}")
        print(f"   - R²:   mean = {r2_mean:.4f}, std = {r2_std:.4f}")
        print(f"   - MAPE: mean = {mape_mean:.4f}%, std = {mape_std:.4f}%")
    
    print(f"\n✅ Semua hasil telah disimpan di folder {base_path}")



🚀 MEMULAI RUN ARIMA KE-1
🔨 Memproses periode Keseluruhan...
💾 Grafik disimpan di saved_models_arima\run_1\plot_keseluruhan.png
🔨 Memproses periode Sebelum COVID...


C:\Users\andil\AppData\Local\Temp\ipykernel_18944\1358872491.py:121: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  wa

💾 Grafik disimpan di saved_models_arima\run_1\plot_sebelum_covid.png
🔨 Memproses periode Selama COVID...


C:\Users\andil\AppData\Local\Temp\ipykernel_18944\1358872491.py:121: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  wa

💾 Grafik disimpan di saved_models_arima\run_1\plot_selama_covid.png
🔨 Memproses periode Setelah COVID...


...

💾 Grafik disimpan di saved_models_arima\run_30\plot_sebelum_covid.png
🔨 Memproses periode Selama COVID...


C:\Users\andil\AppData\Local\Temp\ipykernel_18944\1358872491.py:121: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  wa

💾 Grafik disimpan di saved_models_arima\run_30\plot_selama_covid.png
🔨 Memproses periode Setelah COVID...


C:\Users\andil\AppData\Local\Temp\ipykernel_18944\1358872491.py:121: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  preds.append(f[0])
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  wa

💾 Grafik disimpan di saved_models_arima\run_30\plot_setelah_covid.png

✅ RUN ARIMA 30 SELESAI

🎉 SEMUA 30 RUN ARIMA SELESAI!
💾 Summary metrik disimpan di saved_models_arima\all_runs_summary.json
✅ Plot rata-rata prediksi untuk Keseluruhan disimpan di saved_models_arima\average_results\arima_avg_plot_keseluruhan.png
✅ Plot rata-rata prediksi untuk Sebelum COVID disimpan di saved_models_arima\average_results\arima_avg_plot_sebelum_covid.png
✅ Plot rata-rata prediksi untuk Selama COVID disimpan di saved_models_arima\average_results\arima_avg_plot_selama_covid.png
✅ Plot rata-rata prediksi untuk Setelah COVID disimpan di saved_models_arima\average_results\arima_avg_plot_setelah_covid.png

🔍 Statistik Mean & Standard Deviation untuk ARIMA (30 run):

► Periode Keseluruhan:
   - MSE:  mean = 1.1003, std = 0.0000
   - MAE:  mean = 0.6610, std = 0.0000
   - R²:   mean = 0.9997, std = 0.0000
   - MAPE: mean = 0.6212%, std = 0.0000%

► Periode Sebelum COVID:
   - MSE:  mean = 0.3951, std = 0.0000